# UpSet

> UpSet plot for any number of sets, with every member listed inside its intersection's bar, and the bars packed like a treemap.

In [ ]:
#| default_exp upset_plot

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *
import random, tempfile
from pathlib import Path
from itemiset.examples import load_example

Each exclusive intersection is a bar standing on the x axis, and each member gets one cell in it, so a bar's area is its size. `N = x` under the axis gives each bar's size, and in front of the dot matrix each set's size is in brackets, as in `(76) Dlg`. There is no y axis.

The bars are packed like a treemap: big bars spread over the top of the small ones so that together they fill a rectangle, with as few white cells as possible (usually none, or a few at the top).

In [ ]:
#| export
import heapq, math

import matplotlib
import pandas as pd
from matplotlib.colors import to_rgb
from matplotlib.figure import Figure
from matplotlib.font_manager import FontProperties
from matplotlib.patches import PathPatch, Rectangle
from matplotlib.textpath import TextPath
from shapely.geometry import box
from shapely.ops import unary_union

from itemiset.layout import natural_key
from itemiset.render import _poly_to_path

## Intersections

In [ ]:
#| export
def _members(v) -> list:
    "Members of one set as strings, from a list-like or the first column of a DataFrame."
    return [str(m) for m in (v.iloc[:, 0] if isinstance(v, pd.DataFrame) else v)]

def intersections(sets: dict, sort_by="cardinality") -> list:
    "Exclusive intersections of `sets` as `(set_names, members)` pairs; empty ones are left out."
    names = list(sets)
    where = {}
    for s in names:
        for m in _members(sets[s]):
            where.setdefault(m, set()).add(s)
    zones = {}
    for m, ins in where.items():
        zones.setdefault(tuple(s for s in names if s in ins), []).append(m)
    out = [(z, sorted(ms, key=natural_key)) for z, ms in zones.items()]
    order = {"cardinality": lambda t: (-len(t[1]), len(t[0]), [names.index(s) for s in t[0]]),
             "degree": lambda t: (len(t[0]), -len(t[1]), [names.index(s) for s in t[0]])}
    if sort_by not in order:
        raise ValueError(f"sort_by must be one of {sorted(order)}, not {sort_by!r}")
    return sorted(out, key=order[sort_by])

In [ ]:
demo = {"A": ["a1", "a2", "a3", "ab1", "ab2", "abc"], "B": ["ab1", "ab2", "abc", "b1"], "C": ["abc", "c1", "c2"]}
test_eq(intersections(demo), [(("A",), ["a1", "a2", "a3"]), (("C",), ["c1", "c2"]), (("A", "B"), ["ab1", "ab2"]),
                              (("B",), ["b1"]), (("A", "B", "C"), ["abc"])])
test_eq([z for z, _ in intersections(demo, "degree")], [("A",), ("C",), ("B",), ("A", "B"), ("A", "B", "C")])

## Packing the bars

The bars share a grid of `rows` × `C` cells, and each bar owns a run of columns along the bottom row, its *foot*, left to right in plotting order. Each bar must be one connected piece standing on its foot, and the bars together should fill the grid, with any spare cells in the top row. There are two ways to pack them, and `plan_grid` below keeps whichever looks better.

**Layers** (`pack_layers`): biggest bar first, each bar fills its own columns up to the ceiling and then takes whole rows over the bars to its right, from the top down, so the bars nest like Γ shapes. The spare cells are set aside at the right end of the top row first.

In [ ]:
#| export
def _connected(cells) -> bool:
    "Is `cells` one piece (cells touching side to side)?"
    cells, seen, todo = set(cells), set(), [cells[0]]
    while todo:
        c, r = todo.pop()
        if (c, r) not in seen:
            seen.add((c, r))
            todo += [p for p in ((c + 1, r), (c - 1, r), (c, r + 1), (c, r - 1)) if p in cells]
    return seen == cells

def _width_order(n, N, n_cols, most):
    "Foot widths 1 to `most` for a bar of `n`, closest to its share of the columns first."
    return sorted(range(1, most + 1), key=lambda w: abs(w - n * n_cols / N))

def pack_layers(counts, n_cols, rows, budget=300):
    "Cells `(col, row)` for each bar as nested layers, biggest first, or `None` if no packing is found."
    N, k, tries = sum(counts), len(counts), [budget]
    spare = rows * n_cols - N
    if not 0 <= spare < n_cols or (spare and rows < 2):     # spare cells go in the top row, never the bottom
        return None
    def place(j, a, ceil):               # bars j.. go in columns a..; column c is free below ceil[c]
        if j == k:
            return []
        n = counts[j]
        widths = [n_cols - a] if j == k - 1 else _width_order(n, N, n_cols, min(n, n_cols - a - (k - j - 1)))
        for w in widths:
            for side in (1, -1):         # rows from the left, or from the right
                tries[0] -= 1
                if tries[0] < 0:
                    return None
                foot = [(c, r) for c in range(a, a + w) for r in range(ceil[c])]
                band = sorted(((c, r) for c in range(a + w, n_cols) for r in range(1, ceil[c])),
                              key=lambda t: (-t[1], side * t[0]))[:n - len(foot)]
                if len(foot) > n or len(foot) + len(band) < n or not _connected(foot + band):
                    continue
                lower = list(ceil)
                for c, r in band:
                    lower[c] = min(lower[c], r)
                if (rest := place(j + 1, a + w, lower)) is not None:
                    return [foot + band] + rest
        return None
    return place(0, 0, [rows] * (n_cols - spare) + [rows - 1] * spare)

In [ ]:
def text_grid(cells):
    "Packed bars as text, one letter per bar, top row first."
    C, R = (1 + max(t[i] for cs in cells for t in cs) for i in (0, 1))
    g = [["."] * C for _ in range(R)]
    for i, cs in enumerate(cells):
        for c, r in cs:
            g[r][c] = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"[i]
    return "\n".join("".join(row) for row in reversed(g))

print(text_grid(pack_layers([17, 9, 7, 4, 3, 1], 7, 6)))

Layers can't cope with a long tail of tiny bars: every step down in height has to be covered by a whole row over all the bars to the right, which the tiny bars can't pay for.

**Pouring** (`pack_bars`): the bars are poured in like water, smallest (rightmost) first. A bar starts at its foot and keeps taking the lowest free cell next to what it already has, never flowing left of its foot, so the bigger bars settle over the smaller ones. Within a row it takes the cells furthest right first, or furthest left. A pour is rejected if it seals off free cells the next bar can't reach (they would be white holes), and the search then tries another foot width or side, backing up to earlier bars if it has to. The biggest bar goes in last and tops everything off; with `flat_top=True` its spare cells must all be in the top row.

In [ ]:
#| export
def _pour(n, a, w, rows, C, taken, side):
    "Pour `n` cells in from the foot `[a, a+w)`: lowest row first, then rightmost (`side=-1`) or leftmost (`side=1`)."
    todo, cells, mine = sorted((0, side * c) for c in range(a, a + w)), [], set()
    while todo and len(cells) < n:
        r, c = heapq.heappop(todo)
        c *= side
        if (c, r) in taken or (c, r) in mine:
            continue
        mine.add((c, r))
        cells.append((c, r))
        for c2, r2 in ((c + 1, r), (c - 1, r), (c, r + 1), (c, r - 1)):
            if a <= c2 < C and 0 <= r2 < rows and (c2, r2) not in taken and (c2, r2) not in mine:
                heapq.heappush(todo, (r2, side * c2))
    return cells if len(cells) == n else None

def _reachable(a, rows, C, taken) -> bool:
    "Can every free cell right of column `a` still be reached from column `a - 1`?"
    todo = [(a - 1, r) for r in range(rows)]
    seen = set(todo)
    while todo:
        c, r = todo.pop()
        for p in ((c + 1, r), (c - 1, r), (c, r + 1), (c, r - 1)):
            if a <= p[0] < C and 0 <= p[1] < rows and p not in taken and p not in seen:
                seen.add(p)
                todo.append(p)
    return len(seen) - rows == (C - a) * rows - sum(c >= a for c, _ in taken)

def pack_bars(counts, n_cols, rows, flat_top=True, budget=1000):
    "Cells `(col, row)` for each bar, poured in from the right, or `None` if no packing is found."
    N, tries = sum(counts), [budget]
    def place(i, L, taken):              # bars 0..i still to place, in columns 0..L-1
        if i < 0:
            return []
        n = counts[i]
        widths = _width_order(n, N, n_cols, min(n, L - i))
        if i == 0:                       # the biggest bar takes the columns that are left
            widths = [L] if L <= n else []
        for w in widths:
            for side in ((1,) if i == 0 else (-1, 1)):
                tries[0] -= 1
                if tries[0] < 0:
                    return None
                cells = _pour(n, L - w, w, rows, n_cols, taken, side)
                if cells is None or (i and not _reachable(L - w, rows, n_cols, taken | set(cells))):
                    continue
                taken2 = taken | set(cells)
                top = max(r for _, r in taken2)
                if i == 0 and flat_top and sum(r < top for _, r in taken2) < n_cols * top:
                    continue                 # white cells below the top row
                if (rest := place(i - 1, L - w, taken2)) is not None:
                    return rest + [cells]
        return None
    return place(len(counts) - 1, n_cols, frozenset())

In [ ]:
print(text_grid(pack_bars([45, 37, 22, 19, 11, 9, 8, 5, 4, 4, 4, 2, 2, 2, 1, 1], 18, 10)))

In [ ]:
#| hide
# random bars, both packings: each bar is the right size, in one piece, on a run of the bottom row, in order, with no overlaps
rng = random.Random(0)
for _ in range(300):
    counts = sorted((rng.choice([rng.randint(1, 5), rng.randint(1, 40)]) for _ in range(rng.randint(1, 12))), reverse=True)
    C = rng.randint(len(counts), len(counts) + 10)
    R = math.ceil(sum(counts) / C)
    for cells in (pack_bars(counts, C, R + rng.randint(0, 1), rng.random() < 0.5), pack_layers(counts, C, R)):
        if cells is None: continue
        test_eq(len({c for cs in cells for c in cs}), sum(counts))
        feet = [sorted(c for c, r in cs if r == 0) for cs in cells]
        test_eq(sum(feet, []), list(range(C)))
        for n, cs, f in zip(counts, cells, feet):
            assert len(cs) == n and _connected(cs) and f == list(range(f[0], f[-1] + 1))

## Choosing the grid

`N` members need `N` cells, but the grid can have any shape: 40 could be 4 × 10 or 2 × 20, and a prime like 41 could be 6 × 7 with one cell spare. `plan_grid` tries every number of columns `C` from one per bar upwards, packs the bars both ways into as few rows as it can, and keeps the packing that scores best:

- close to `target_aspect` (measured in inches, so long names make wide cells);
- few white cells, with white cells below the top row counting double;
- tidy shapes (short outlines), to break near-ties.

In [ ]:
#| export
def _outline(cells) -> int:
    "Length of the outline of `cells`, in cell sides."
    s = set(cells)
    return sum((c + dc, r + dr) not in s for c, r in s for dc, dr in ((1, 0), (-1, 0), (0, 1), (0, -1)))

def plan_grid(counts, cell_aspect, target_aspect=1.6, white_weight=10.0, shape_weight=0.5):
    "Foot widths, rows and cells for bars of `counts` cells, each cell `cell_aspect` times wider than tall."
    N, best = sum(counts), None
    for C in range(len(counts), N + 1):
        R0 = math.ceil(N / C)
        poured = pack_bars(counts, C, R0) or pack_bars(counts, C, R0, False) or pack_bars(counts, C, R0 + 1, False)
        for cells in filter(None, (pack_layers(counts, C, R0), poured)):
            R = 1 + max(r for cs in cells for _, r in cs)
            low_white = C * (R - 1) - sum(r < R - 1 for cs in cells for _, r in cs)
            ragged = sum(map(_outline, cells)) / sum(2 * n + 2 for n in counts) - 1
            score = (abs(math.log(C * cell_aspect / R / target_aspect))
                     + white_weight * (R * C - N + low_white) / (R * C) + shape_weight * ragged)
            if best is None or score < best[0]:
                best = (score, [sum(r == 0 for _, r in cs) for cs in cells], R, cells)
        if best and C * cell_aspect / R0 > 4 * target_aspect:      # it only gets wider from here
            break
    return best[1:]

In [ ]:
widths, R, cells = plan_grid([20, 12, 12, 10, 10, 10], cell_aspect=5.8)
print(widths, R)
print(text_grid(cells))

## `upset`

In [ ]:
#| export
MEMBER_FONT = "DejaVu Sans Mono"

def _text_w(s, size, family=None, weight="normal") -> float:
    "Width of `s` in inches at `size` points."
    return TextPath((0, 0), s, prop=FontProperties(family=family, size=size, weight=weight)).get_extents().width / 72

def _lum(c) -> float:
    r, g, b = to_rgb(c)
    return 0.2126 * r + 0.7152 * g + 0.0722 * b

In [ ]:
#| export
def upset(data_dict, color_list=None, sort_by="cardinality", target_aspect=1.6, fontsize=8,
          count_fontsize=10, gridkey_label_fontsize=10, circle_size=100, corner=0.3, matrix_height=1.0, font=None,
          title="", title_fontsize=20, title_fontweight="bold", save_path=None) -> Figure:
    "UpSet plot of `{name: members}`, with each member in a cell of its intersection's bar."
    names = list(data_dict)
    zones = intersections(data_dict, sort_by)
    counts, k = [len(ms) for _, ms in zones], len(zones)
    if isinstance(color_list, str):
        cmap = matplotlib.colormaps[color_list]
        base = cmap.colors if hasattr(cmap, "colors") else [cmap(i / max(k - 1, 1)) for i in range(k)]
        color_list = [base[i % len(base)] for i in range(k)]
    if color_list is not None and len(color_list) != k:
        raise ValueError(f"Number of colors given ({len(color_list)}) does not match the number of bars ({k})")
    colours = color_list or ["#222222"] * k

    # sizes in inches
    longest = max((m for _, ms in zones for m in ms), key=len)
    cw, ch = _text_w(longest, fontsize, MEMBER_FONT) + 0.9 * fontsize / 72, 1.7 * fontsize / 72
    widths, R, cells = plan_grid(counts, cw / ch, target_aspect)
    C, starts = sum(widths), [sum(widths[:i]) for i in range(k)]
    xs = [(a + w / 2) * cw for a, w in zip(starts, widths)]
    texts = [f"N = {n}" for n in counts]
    count_fs = min([count_fontsize] + [count_fontsize * 0.8 * w * cw / _text_w(t, count_fontsize, font)
                                       for t, w in zip(texts, widths)])     # shrink to fit under the narrowest foot
    sizes = [f"({len(set(_members(data_dict[s])))})" for s in names]
    size_w, name_w = (max(_text_w(t, gridkey_label_fontsize, font) for t in ts) for ts in (sizes, names))
    lab_w = size_w + name_w + 0.5
    row_h = 0.32 * matrix_height * gridkey_label_fontsize / 13
    title_h = title_fontsize / 72 * 1.8 if title else 0
    y_mat = -count_fs / 72 * 2.2                       # top of the dot matrix
    left, right = -lab_w, max(C * cw + 0.25, -lab_w + (_text_w(title, title_fontsize, font, title_fontweight) + 0.3 if title else 0))
    top, bottom = R * ch + 0.15 + title_h, y_mat - len(names) * row_h - 0.1

    # one axes over the whole figure, in inches
    fig = Figure(figsize=(right - left, top - bottom))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis("off")
    ax.set_xlim(left, right)
    ax.set_ylim(bottom, top)

    # bars: the union of their cells, shrunk a little so neighbours stay apart; members read row by row
    for i, (cs, (_, ms)) in enumerate(zip(cells, zones)):
        shape = unary_union([box(c * cw, r * ch, (c + 1) * cw, (r + 1) * ch) for c, r in cs]).buffer(-0.07 * ch, join_style="mitre")
        r = corner * ch                  # round the corners, inside and out, like the itemiset plots
        shape = shape.buffer(r, join_style="round").buffer(-2 * r, join_style="round").buffer(r, join_style="round")
        ax.add_patch(PathPatch(_poly_to_path(shape), facecolor=colours[i], edgecolor="none", zorder=1))
        tc = "white" if _lum(colours[i]) < 0.5 else "#111111"
        for (c, r), m in zip(sorted(cs, key=lambda t: (-t[1], t[0])), ms):
            ax.text((c + 0.5) * cw, (r + 0.5) * ch, m, ha="center", va="center", fontsize=fontsize,
                    family=MEMBER_FONT, color=tc, zorder=3)

    # x axis, with "N = x" under each bar's foot
    ax.plot([0, C * cw], [0, 0], color="#333333", lw=0.8, solid_capstyle="butt")
    for x, t in zip(xs, texts):
        ax.plot([x, x], [0, -0.04], color="#333333", lw=0.8)
        ax.text(x, y_mat / 2, t, ha="center", va="center", fontsize=count_fs, family=font, color="#333333")

    # dot matrix, with "(size) name" in front of each set's row
    for j, (s, size) in enumerate(zip(names, sizes)):
        yc = y_mat - (j + 0.5) * row_h
        if j % 2 == 0:
            ax.add_patch(Rectangle((left + 0.1, yc - row_h / 2), C * cw + lab_w - 0.1, row_h, facecolor="#F2F2F2",
                                   edgecolor="none", zorder=0))
        ax.text(size_w - lab_w + 0.25, yc, size, ha="right", va="center", fontsize=gridkey_label_fontsize, family=font)
        ax.text(-0.1, yc, s, ha="right", va="center", fontsize=gridkey_label_fontsize, family=font)
        ax.scatter(xs, [yc] * k, s=circle_size, linewidths=0, zorder=3,
                   color=[colours[i] if s in z else "#D6D6D6" for i, (z, _) in enumerate(zones)])
    for i, ((z, _), x) in enumerate(zip(zones, xs)):
        ys = [y_mat - (names.index(s) + 0.5) * row_h for s in z]
        ax.plot([x, x], [min(ys), max(ys)], color=colours[i], lw=0.25 * circle_size ** 0.5, zorder=2)

    if title:
        ax.text(left + 0.15, top - 0.05, title, ha="left", va="top", fontsize=title_fontsize,
                fontweight=title_fontweight, family=font)
    if save_path:
        with matplotlib.rc_context({"svg.fonttype": "none", "pdf.fonttype": 42}):
            fig.savefig(save_path, dpi=300)
    return fig

In [ ]:
ex = load_example("pbody")
upset(ex["sets"], color_list="Set2", title="P-body proximity labelling")

### Arguments

| Argument | Controls |
|---|---|
| `color_list` | bar colours, one per bar, or a colormap name (e.g. `"Set2"`); default black bars with white names |
| `sort_by` | bar order: `"cardinality"` (biggest first) or `"degree"` (fewest sets first) |
| `target_aspect` | width ÷ height of the block of bars that the grid aims for |
| `fontsize` | member font size; the figure grows to fit |
| `count_fontsize` | largest size of the `N = x` counts; they shrink to fit under the narrowest bar |
| `gridkey_label_fontsize` | size of the set labels: the `(size)` column and the names |
| `circle_size`, `matrix_height` | dot size and dot-matrix row height |
| `corner` | corner radius of the bars, as a fraction of the row height |
| `font` | font for counts, set labels and title (member names are monospace) |
| `title`, `title_fontsize`, `title_fontweight` | the title |
| `save_path` | also save to this file (.png, .pdf or .svg; text stays live in SVG and PDF) |

In [ ]:
tmp = Path(tempfile.mkdtemp())
upset(demo, save_path=tmp/"demo.svg")
svg = (tmp/"demo.svg").read_text()
assert ">ab1</text>" in svg and "N = 3" in svg and ">(6)</text>" in svg
test_fail(lambda: upset(demo, color_list=["red"]), contains="does not match")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()